In [1]:
# Parameters
BATCH_MODE = True


# Jeu de devinette : Père Fouras vs Laurent Jalabert

**Navigation** : [Index](../README.md)

Dans ce notebook, nous allons simuler le duel légendaire entre le Père Fouras et Laurent Jalabert en utilisant Semantic Kernel avec des agents conversationnels.


In [2]:
# Import guards - availability flags for external dependencies

try:
    from dotenv import load_dotenv
    DOTENV_AVAILABLE = True
except ImportError:
    DOTENV_AVAILABLE = False
    print(f'  dotenv non disponible - certaines fonctionnalites seront limitees')

try:
    import semantic_kernel
    SEMANTIC_KERNEL_AVAILABLE = True
except ImportError:
    SEMANTIC_KERNEL_AVAILABLE = False
    print(f'  semantic_kernel non disponible - certaines fonctionnalites seront limitees')


# Bloc 1 - Installation et imports
%pip install semantic-kernel python-dotenv --quiet
import os
import logging
from dotenv import load_dotenv
from semantic_kernel import Kernel
from semantic_kernel.agents import ChatCompletionAgent, AgentGroupChat
from semantic_kernel.agents.strategies import KernelFunctionTerminationStrategy
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import ChatHistory
from semantic_kernel.functions import KernelArguments

# Configuration des logs
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger('FortBoyard')

# Chargement des variables d'environnement
env_path = os.path.join(os.path.dirname(os.path.abspath(".")), ".env")
if os.path.exists(env_path):
    load_dotenv(env_path)
else:
    load_dotenv()



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Configuration des agents

In [3]:
# Bloc 2 - Création du kernel
MOT_A_DEVINER = "anticonstitutionnellement"

def create_kernel():
    kernel = Kernel()
    model_id = os.getenv("OPENAI_CHAT_MODEL_ID", "gpt-5-mini")
    kernel.add_service(OpenAIChatCompletion(
        service_id="openai",
        ai_model_id=model_id,
        api_key=os.getenv("OPENAI_API_KEY")
    ))
    return kernel


## Définition des prompts

In [4]:
# Bloc 3 - Prompts des agents
PERE_FOURAS_PROMPT = f"""
Tu es le Père Fouras de Fort Boyard. 
Tu dois faire deviner le mot '{MOT_A_DEVINER}'. 
Utilise des charades et réponses énigmatiques. 
Ne révèle jamais directement le mot !
"""

LAURENT_JALABERT_PROMPT = """
Tu es Laurent Jalabert. 
Tu dois deviner le mot en posant des questions fermées (Oui/Non).
Sois perspicace et stratégique dans tes questions.
"""

## Création des agents avec stratégies personnalisées

In [5]:
# Bloc 4 - Définition des agents
pere_fouras = ChatCompletionAgent(
    kernel=create_kernel(),
    name="Pere_Fouras",
    instructions=PERE_FOURAS_PROMPT,
)

laurent_jalabert = ChatCompletionAgent(
    kernel=create_kernel(),
    name="Laurent_Jalabert",
    instructions=LAURENT_JALABERT_PROMPT,
)

## Stratégie de terminaison personnalisée

In [6]:
from semantic_kernel.agents.strategies.termination.termination_strategy import TerminationStrategy
from semantic_kernel.contents.chat_message_content import ChatMessageContent

# Bloc 5 - Logique de terminaison
class FortBoyardTerminationStrategy(TerminationStrategy):
    """Arrête la partie si le mot est deviné"""
    
    async def should_agent_terminate(
        self, 
        agent: ChatCompletionAgent, 
        history: list[ChatMessageContent], 
        cancellation_token = None
    ) -> bool:
        if not history:
            return False
        
        last_message = str(history[-1].content).lower()
        return MOT_A_DEVINER in last_message

## Configuration du groupe de discussion

In [7]:
# Bloc 6 - Configuration corrigée
chat = AgentGroupChat(
    agents=[pere_fouras, laurent_jalabert],
    termination_strategy=FortBoyardTerminationStrategy(
        agents=[laurent_jalabert],  # Définit explicitement les agents
        maximum_iterations=20       # Définit le nombre max d'itérations
    )
)


## Lancement de la partie !

In [8]:
from semantic_kernel.contents import AuthorRole, ChatMessageContent

# Bloc 7 - Exécution du jeu
async def jouer_partie():
    logger.info("Depart du duel Pere Fouras vs Laurent Jalabert !")
    logger.info(f"Mot a deviner : {MOT_A_DEVINER.upper()}")
    
    # Message initial pour lancer la conversation
    await chat.add_chat_message(
        ChatMessageContent(role=AuthorRole.USER, content="Pere Fouras, donne-moi un indice !")
    )
    
    async for message in chat.invoke():
        role = message.role
        logger.info(f"[{role}] : {message.content}")
    
    logger.info("Partie terminee !")

await jouer_partie()

2026-05-13 02:23:40,910 [INFO] Depart du duel Pere Fouras vs Laurent Jalabert !


2026-05-13 02:23:40,911 [INFO] Mot a deviner : ANTICONSTITUTIONNELLEMENT


2026-05-13 02:23:40,912 [INFO] Adding `1` agent chat messages


2026-05-13 02:23:40,913 [INFO] Selected agent at index 0 (ID: 84766105-9a1b-4e7d-8d69-5927eb188f3b, name: Pere_Fouras)


2026-05-13 02:23:40,913 [INFO] Invoking agent Pere_Fouras


...

2026-05-13 02:23:54,911 [INFO] OpenAI usage: CompletionUsage(completion_tokens=19, prompt_tokens=1023, total_tokens=1042, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


2026-05-13 02:23:54,912 [INFO] Evaluating termination criteria for 3bf0ebff-94d5-4c86-98f6-11a128686a02


2026-05-13 02:23:54,913 [INFO] Evaluated criteria for 3bf0ebff-94d5-4c86-98f6-11a128686a02, should terminate: True


2026-05-13 02:23:54,914 [INFO] [AuthorRole.ASSISTANT] : Ton indice pointe très clairement vers : **« anticonstitutionnellement »**.


2026-05-13 02:23:54,915 [INFO] Partie terminee !
